# Module 2 — Evaluation & Analysis

Five mandatory evaluation protocols: topology map, circuit overlap, layer evolution, compositionality, and marginalization robustness.

In [ ]:
# Cell 1 – Setup
import subprocess, sys, os, shutil
for pkg in ["h5py", "umap-learn", "seaborn", "matplotlib", "numpy", "pandas", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Mount Drive ──────────────────────────────────────────────────────────
from google.colab import drive
mp = "/content/drive"
if os.path.isdir(mp) and os.listdir(mp):
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    shutil.rmtree(mp, ignore_errors=True)
drive.mount(mp)

# ── Paths ────────────────────────────────────────────────────────────────
ATLAS_HDF5 = "/content/drive/MyDrive/DATA/CSP-Atlas/dynamic_feature_atlas.h5"
STATS_JSON = "/content/drive/MyDrive/DATA/CSP-Atlas/extraction_stats.json"

LOCAL_SRC = "/Users/piotrwilam/CODE/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, h5py, json
from module2.io_utils import load_atlas_hdf5
from module2.metrics  import jaccard_similarity, entanglement_index

atlas = load_atlas_hdf5(ATLAS_HDF5)
pair_masks       = atlas["pair_masks"]
universal_masks  = atlas["universal_masks"]
metrics          = atlas["metrics"]
metadata         = atlas["metadata"]

print("Atlas loaded")
print(f"  Pairs          : {len(pair_masks)}")
print(f"  Universal AST  : {len(universal_masks['ast'])}")
print(f"  Universal Blt  : {len(universal_masks['builtin'])}")
print(f"  Metadata       : {metadata}")

## Protocol 1: Topology Map (UMAP)

In [ ]:
# Cell 2 – Protocol 1: Topology Map (UMAP) — Universal AST modules
import umap, numpy as np, matplotlib.pyplot as plt

REP_LAYER  = 4
ast_names  = [n for n in sorted(universal_masks["ast"]) if REP_LAYER in universal_masks["ast"][n]]
ast_vecs   = np.array([universal_masks["ast"][n][REP_LAYER].astype(np.float32) for n in ast_names])

reducer = umap.UMAP(metric="jaccard", n_neighbors=min(10, len(ast_names)-1),
                    min_dist=0.1, random_state=42, verbose=False)
emb = reducer.fit_transform(ast_vecs)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(emb[:, 0], emb[:, 1], s=30, alpha=0.7)
for i, name in enumerate(ast_names):
    ax.annotate(name, (emb[i, 0], emb[i, 1]), fontsize=6, ha="center", va="bottom")
ax.set_title("UMAP — Universal AST Modules (Jaccard metric, layer 4)")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
plt.tight_layout(); plt.show()
print(f"Plotted {len(emb)} AST modules")


In [ ]:
# Cell 3 – Protocol 1 continued: UMAP for Universal Builtin modules
import umap, numpy as np, matplotlib.pyplot as plt

blt_names = [n for n in sorted(universal_masks["builtin"]) if REP_LAYER in universal_masks["builtin"][n]]
blt_vecs  = np.array([universal_masks["builtin"][n][REP_LAYER].astype(np.float32) for n in blt_names])

reducer_b = umap.UMAP(metric="jaccard", n_neighbors=min(10, len(blt_names)-1),
                       min_dist=0.1, random_state=42, verbose=False)
emb_b = reducer_b.fit_transform(blt_vecs)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(emb_b[:, 0], emb_b[:, 1], s=30, alpha=0.7, color="orange")
for i, name in enumerate(blt_names):
    ax.annotate(name, (emb_b[i, 0], emb_b[i, 1]), fontsize=6, ha="center", va="bottom")
ax.set_title("UMAP — Universal Builtin Modules (Jaccard metric, layer 4)")
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
plt.tight_layout(); plt.show()


## Protocol 2: Circuit Overlap (Jaccard Heatmaps)

In [ ]:
# Cell 4 – Protocol 2: Circuit Overlap (Jaccard heatmaps)
import matplotlib.pyplot as plt, seaborn as sns, numpy as np

def plot_jaccard_heatmap(mat, names, title, max_labels=50):
    fig, ax = plt.subplots(figsize=(min(24, len(names)*0.35+2),
                                    min(24, len(names)*0.35+2)))
    show = names if len(names) <= max_labels else False
    sns.heatmap(mat, ax=ax, vmin=0, vmax=1, cmap="viridis",
                xticklabels=show, yticklabels=show)
    ax.set_title(title)
    plt.tight_layout(); plt.show()
    off_diag = mat[np.triu_indices_from(mat, k=1)]
    print(f"{title}")
    print(f"  Mean Jaccard: {off_diag.mean():.4f}")
    print(f"  Max  Jaccard: {off_diag.max():.4f}")
    print(f"  Pairs with J>0.5: {(off_diag > 0.5).sum()}")

if "jaccard_ast_matrix" in metrics:
    names_m = metrics.get("ast_names", sorted(universal_masks["ast"]))
    plot_jaccard_heatmap(metrics["jaccard_ast_matrix"], names_m,
                         "Jaccard Similarity — Universal AST Modules (layer 4)")

if "jaccard_builtin_matrix" in metrics:
    names_m = metrics.get("builtin_names", sorted(universal_masks["builtin"]))
    plot_jaccard_heatmap(metrics["jaccard_builtin_matrix"], names_m,
                         "Jaccard Similarity — Universal Builtin Modules (layer 4)")


## Protocol 3: Layer Evolution

In [ ]:
# Cell 5 – Protocol 3: Layer Evolution
import numpy as np, matplotlib.pyplot as plt

layer_ids = sorted({lid for lm in pair_masks.values() for lid in lm})

def mean_size(masks_dict, lid):
    s = [lm[lid].sum() for lm in masks_dict.values() if lid in lm]
    return np.mean(s) if s else 0

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(layer_ids, [mean_size(pair_masks, l) for l in layer_ids],
        marker="o", label="Pair Representations")
ax.plot(layer_ids, [mean_size(universal_masks["ast"], l) for l in layer_ids],
        marker="s", label="Universal AST")
ax.plot(layer_ids, [mean_size(universal_masks["builtin"], l) for l in layer_ids],
        marker="^", label="Universal Builtin")
ax.set_xlabel("Layer"); ax.set_ylabel("Mean circuit size (neurons)")
ax.set_title("Circuit size evolution across layers")
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()


## Protocol 4: Compositionality (Entanglement Index)

In [ ]:
# Cell 6 – Protocol 4: Compositionality (Entanglement Index)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from module2.metrics import entanglement_index

REP_LAYER = 4
ei_rows = []

for (ast_n, blt_o), lm in pair_masks.items():
    if REP_LAYER not in lm:
        continue
    pm = lm[REP_LAYER]
    am = universal_masks["ast"].get(ast_n, {}).get(REP_LAYER)
    bm = universal_masks["builtin"].get(blt_o, {}).get(REP_LAYER)
    if am is None or bm is None:
        continue
    ei_rows.append({
        "ast_node": ast_n, "builtin_obj": blt_o,
        "E_I": entanglement_index(pm, am, bm),
        "pair_size": int(pm.sum()),
    })

ei_df = pd.DataFrame(ei_rows)
print(f"E_I computed for {len(ei_df)} pairs at layer {REP_LAYER}")
print(ei_df["E_I"].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(ei_df["E_I"].values, bins=40, edgecolor="black")
axes[0].set_xlabel("Entanglement Index (E_I)")
axes[0].set_ylabel("Count")
axes[0].set_title("E_I distribution\nE_I=0: compositional | E_I=1: unique")

axes[1].scatter(ei_df["pair_size"], ei_df["E_I"], alpha=0.4, s=20)
axes[1].set_xlabel("Pair circuit size")
axes[1].set_ylabel("E_I")
axes[1].set_title("Circuit size vs. E_I")
plt.tight_layout(); plt.show()


In [ ]:
# Cell 7 – Protocol 4 continued: most/least compositional pairs
print("Most compositional pairs (lowest E_I):")
print(ei_df.nsmallest(10, "E_I")[["ast_node","builtin_obj","E_I"]].to_string(index=False))
print("\nLeast compositional pairs (highest E_I):")
print(ei_df.nlargest(10, "E_I")[["ast_node","builtin_obj","E_I"]].to_string(index=False))


## Protocol 5: Marginalization Robustness

In [ ]:
# Cell 8 – Protocol 5: Marginalization Robustness — AST
import numpy as np, matplotlib.pyplot as plt, random

random.seed(42)
sample_ast = random.sample(sorted(universal_masks["ast"]), min(5, len(universal_masks["ast"])))

fig, ax = plt.subplots(figsize=(9, 5))
for ast_n in sample_ast:
    blts = [b for (a, b) in pair_masks if a == ast_n and REP_LAYER in pair_masks[(a, b)]]
    if len(blts) < 2:
        continue
    random.shuffle(blts)
    running, sizes = None, []
    for blt in blts:
        pm = pair_masks[(ast_n, blt)][REP_LAYER]
        running = pm.copy() if running is None else np.logical_and(running, pm)
        sizes.append(int(running.sum()))
    ax.plot(range(1, len(sizes)+1), sizes, marker="o", label=ast_n, alpha=0.8)

ax.set_xlabel("Number of builtins intersected")
ax.set_ylabel("Universal AST circuit size")
ax.set_title("Marginalization Robustness — Universal AST convergence")
ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 9 – Protocol 5 continued: Builtin marginalization robustness
import numpy as np, matplotlib.pyplot as plt, random

sample_blt = random.sample(sorted(universal_masks["builtin"]), min(5, len(universal_masks["builtin"])))

fig, ax = plt.subplots(figsize=(9, 5))
for blt_o in sample_blt:
    asts = [a for (a, b) in pair_masks if b == blt_o and REP_LAYER in pair_masks[(a, b)]]
    if len(asts) < 2:
        continue
    random.shuffle(asts)
    running, sizes = None, []
    for ast_n in asts:
        pm = pair_masks[(ast_n, blt_o)][REP_LAYER]
        running = pm.copy() if running is None else np.logical_and(running, pm)
        sizes.append(int(running.sum()))
    ax.plot(range(1, len(sizes)+1), sizes, marker="s", label=blt_o, alpha=0.8)

ax.set_xlabel("Number of AST nodes intersected")
ax.set_ylabel("Universal Builtin circuit size")
ax.set_title("Marginalization Robustness — Universal Builtin convergence")
ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout(); plt.show()


## Ockham Index (O_I)

In [ ]:
# Cell 10 – Ockham Index (Jaccard distance between Universal Modules)
from module2.metrics import jaccard_distance
import pandas as pd

ast_sorted = sorted(universal_masks["ast"])
oi_rows = []
for i, a1 in enumerate(ast_sorted):
    m1 = universal_masks["ast"][a1].get(REP_LAYER)
    if m1 is None: continue
    for a2 in ast_sorted[i+1:]:
        m2 = universal_masks["ast"][a2].get(REP_LAYER)
        if m2 is None: continue
        oi_rows.append({"a": a1, "b": a2, "O_I": jaccard_distance(m1, m2)})

oi_df = pd.DataFrame(oi_rows)
print("Ockham Index (AST x AST) — summary:")
print(oi_df["O_I"].describe())
print("\nMost similar AST node pairs (low O_I = high overlap):")
print(oi_df.nsmallest(10, "O_I")[["a","b","O_I"]].to_string(index=False))


## Summary Report

In [ ]:
# Cell 11 – Summary report
import json, datetime

report = {
    "generated_at"        : datetime.datetime.utcnow().isoformat() + "Z",
    "n_pairs"             : len(pair_masks),
    "n_universal_ast"     : len(universal_masks["ast"]),
    "n_universal_builtin" : len(universal_masks["builtin"]),
    "mean_EI"             : float(ei_df["E_I"].mean()) if len(ei_df) else None,
    "fraction_EI_lt_0.2"  : float((ei_df["E_I"] < 0.2).mean()) if len(ei_df) else None,
    "mean_OI_ast"         : float(oi_df["O_I"].mean()) if len(oi_df) else None,
    "metadata"            : {str(k): str(v) for k, v in metadata.items()},
}

print(json.dumps(report, indent=2))

# Close atlas HDF5 handle
handle = atlas.get("handle")
if handle and handle.id.valid:
    handle.close()
    print("HDF5 handle closed.")


In [ ]:
# Cell 12 – Done
print("Module 2 evaluation complete.")
print("All 5 protocols executed:")
print("  1. Topology Map (UMAP)")
print("  2. Circuit Overlap (Jaccard heatmaps)")
print("  3. Layer Evolution")
print("  4. Compositionality (Entanglement Index)")
print("  5. Marginalization Robustness")
